# Oral Lesion Classification with MobileNetV3-Small
### Benign vs malignant on smartphone and intraoral photos, with leakage-safe cross-validation, calibration and Grad-CAM

This trains the model that the oral lesion surveillance app was designed for (MobileNetV3-Small, small enough to run on a phone).

**Dataset:** Oral Images Dataset, Mendeley Data, version 2 (CC BY 4.0). 165 benign and 158 malignant lesion photos from hospitals in Karnataka.
Citation: Chandrashekar H S, Geetha Kiran A, Murali S, Dinesh M S, Nanditha B R (2021). *Oral Images Dataset*. Mendeley Data. https://doi.org/10.17632/mhjyrn35p4.2

**How to run (Google Colab):**
1. `Runtime → Change runtime type → T4 GPU`
2. Open https://data.mendeley.com/datasets/mhjyrn35p4/2 and click **Download All**. Keep the zip as it is.
3. Run the cells top to bottom. Cell 3 asks you to upload the zip.

**Time on a T4 GPU:** about 10–15 minutes.

**Why these choices (you will be asked):**
- Only `original_data` is used. The `augmented_data` folder holds flipped and rotated copies of the same photos; using it would put copies of one lesion in both training and test sets.
- The dataset has no patient IDs, so near-identical photos are grouped by image hash and kept in the same fold. This is the closest we can get to a patient-level split.
- With only 323 images, a single test split is noisy, so we use 5-fold cross-validation and report every image's out-of-fold prediction, with a bootstrap 95% confidence interval.

## 1. Check the GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Go to Runtime -> Change runtime type -> T4 GPU, then rerun.")

## 2. Install libraries

In [ ]:
!pip install -q timm grad-cam

## 3. Upload the dataset zip
Click **Choose Files** and select the zip you downloaded from Mendeley. Uploading can take a few minutes.
If you already put the zip in Google Drive, set `DRIVE_ZIP` to its path instead and skip the upload.

In [ ]:
import os, zipfile, glob, shutil

DRIVE_ZIP = ""          # e.g. "/content/drive/MyDrive/oral_images.zip" (leave empty to upload)
DATA_ROOT = "/content/oral_data"
os.makedirs(DATA_ROOT, exist_ok=True)

if DRIVE_ZIP:
    from google.colab import drive
    drive.mount("/content/drive")
    zips = [DRIVE_ZIP]
else:
    from google.colab import files
    uploaded = files.upload()
    zips = [os.path.join("/content", n) for n in uploaded]

def extract_all(zpaths, dest):
    # Mendeley's "Download All" zip can contain further zips inside; unpack every level.
    queue = list(zpaths)
    seen = set()
    while queue:
        z = queue.pop()
        if z in seen:
            continue
        seen.add(z)
        out = os.path.join(dest, os.path.splitext(os.path.basename(z))[0])
        with zipfile.ZipFile(z) as f:
            f.extractall(out)
        queue += glob.glob(os.path.join(out, "**", "*.zip"), recursive=True)

extract_all(zips, DATA_ROOT)
print("Extracted. Top-level folders:", os.listdir(DATA_ROOT))

## 4. Collect the original images and their labels
Labels come from the folder names (`benign` / `malignant`). Augmented copies are excluded.
You should see about **165 benign and 158 malignant**.

In [ ]:
import os, pandas as pd

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

def collect(root):
    rows = []
    for dirpath, _, filenames in os.walk(root):
        low = dirpath.lower()
        if "augment" in low or "__macosx" in low:
            continue
        parts = [p.lower() for p in dirpath.split(os.sep)]
        label = None
        for p in reversed(parts):
            if "malignant" in p or "cancer" in p and "non" not in p:
                label = 1; break
            if "benign" in p or "non" in p and "cancer" in p:
                label = 0; break
        if label is None:
            continue
        for fn in filenames:
            if fn.lower().endswith(IMG_EXT) and not fn.startswith("._"):
                rows.append({"path": os.path.join(dirpath, fn), "label": label})
    return pd.DataFrame(rows)

df = collect(DATA_ROOT)
has_original = df["path"].str.lower().str.contains("original").any()
if has_original:
    df = df[df["path"].str.lower().str.contains("original")].reset_index(drop=True)
print("Images found:", len(df))
print(df["label"].map({0: "benign", 1: "malignant"}).value_counts())
assert len(df) > 0, "No images found. Check the folder names printed in cell 3."

## 5. Group near-duplicate photos
Some lesions may have been photographed more than once. Identical or near-identical photos get the same group ID,
so cross-validation never puts one in training and its twin in testing.

In [ ]:
import numpy as np
from PIL import Image

def dhash(path, size=8):
    img = Image.open(path).convert("L").resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(img, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

hashes = np.stack([dhash(p) for p in df["path"]])
n = len(df)
parent = list(range(n))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i in range(n):
    d = (hashes[i + 1:] != hashes[i]).sum(1)
    for j in np.where(d <= 4)[0] + i + 1:      # <= 4 of 64 bits differ: near-duplicate
        parent[find(j)] = find(i)
df["group"] = [find(i) for i in range(n)]
dup = n - df["group"].nunique()
print(f"{n} images in {df['group'].nunique()} groups ({dup} near-duplicates grouped with another photo)")

## 6. Settings, dataset and augmentation

In [ ]:
import random, torch, timm, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

SEED = 42
N_FOLDS = 5
EPOCHS = 25
IMG = 224
BATCH = 32
LR = 3e-4
MODEL_NAME = "mobilenetv3_small_100"     # MobileNetV3-Small, ~1.5M parameters, ImageNet-pretrained
PRETRAINED = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([
    T.RandomResizedCrop(IMG, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(0.15, 0.15, 0.1, 0.02),   # mild: lesion colour is a real signal, so we do not distort it much
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((IMG, IMG)), T.ToTensor(), T.Normalize(MEAN, STD)])

class OralDS(Dataset):
    def __init__(self, frame, tf):
        self.p, self.y, self.tf = frame["path"].tolist(), frame["label"].tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert("RGB")), torch.tensor(self.y[i], dtype=torch.float32)

def make_model():
    m = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=1)
    return m.to(DEVICE)
print("Device:", DEVICE)

## 7. Five-fold cross-validation
In every fold: train on 4/5 of the groups, pick the best epoch and the decision threshold (Youden's J) on a validation slice of the
training groups, then predict the held-out fifth. Every image gets exactly one prediction from a model that never saw it.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve

@torch.no_grad()
def predict(model, frame):
    model.eval()
    dl = DataLoader(OralDS(frame, eval_tf), batch_size=BATCH, shuffle=False, num_workers=2)
    out = []
    for x, _ in dl:
        with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
            out.append(torch.sigmoid(model(x.to(DEVICE)).float().squeeze(1)).cpu())
    return torch.cat(out).numpy()

def youden(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr - fpr)])

outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
df["oof_prob"] = np.nan; df["fold"] = -1; df["fold_thr"] = np.nan
fold_rows, models = [], []

for k, (tr_idx, te_idx) in enumerate(outer.split(df, df["label"], df["group"])):
    seed_all(SEED + k)
    tr_all, te = df.iloc[tr_idx], df.iloc[te_idx]
    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED + k)
    fit_i, val_i = next(inner.split(tr_all, tr_all["label"], tr_all["group"]))
    fit, val = tr_all.iloc[fit_i], tr_all.iloc[val_i]

    model = make_model()
    pos = fit["label"].sum(); neg = len(fit) - pos
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / max(pos, 1), device=DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
    dl = DataLoader(OralDS(fit, train_tf), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=len(fit) > BATCH)

    best_auc, best_state = -1, None
    for ep in range(EPOCHS):
        model.train()
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
                loss = loss_fn(model(x).squeeze(1), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        vp = predict(model, val)
        vauc = roc_auc_score(val["label"], vp) if val["label"].nunique() > 1 else 0.5
        if vauc > best_auc:
            best_auc, best_state = vauc, {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    thr = youden(val["label"], predict(model, val)) if val["label"].nunique() > 1 else 0.5
    p = predict(model, te)
    df.loc[df.index[te_idx], "oof_prob"] = p
    df.loc[df.index[te_idx], "fold"] = k
    df.loc[df.index[te_idx], "fold_thr"] = thr
    pred = (p >= thr).astype(int); yt = te["label"].values
    tp = ((pred == 1) & (yt == 1)).sum(); fn = ((pred == 0) & (yt == 1)).sum()
    tn = ((pred == 0) & (yt == 0)).sum(); fp = ((pred == 1) & (yt == 0)).sum()
    fold_rows.append({"fold": k + 1, "n_test": len(te), "val_auc": round(best_auc, 3),
                      "test_auc": round(roc_auc_score(yt, p), 3), "threshold": round(thr, 3),
                      "sensitivity": round(float(tp / max(tp + fn, 1)), 3), "specificity": round(float(tn / max(tn + fp, 1)), 3)})
    models.append(model)
    print(fold_rows[-1])

folds = pd.DataFrame(fold_rows)
folds

## 8. Results on all 323 out-of-fold predictions
- **AUC** with a bootstrap 95% confidence interval (1,000 resamples)
- **Sensitivity and specificity** at each fold's own validation-chosen threshold
- **ECE** (expected calibration error, 10 bins): is "80% sure" right about 80% of the time?

In [ ]:
from sklearn.metrics import confusion_matrix

y = df["label"].values; p = df["oof_prob"].values
pred = (p >= df["fold_thr"].values).astype(int)
tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
auc = roc_auc_score(y, p)

rng = np.random.default_rng(SEED)
boots = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    if len(np.unique(y[i])) == 2:
        boots.append(roc_auc_score(y[i], p[i]))
lo, hi = np.percentile(boots, [2.5, 97.5])

def ece_score(y, p, bins=10):
    conf = np.where(p >= 0.5, p, 1 - p); correct = ((p >= 0.5).astype(int) == y)
    edges = np.linspace(0.5, 1.0, bins + 1); e = 0.0
    for a, b in zip(edges[:-1], edges[1:]):
        m = (conf > a) & (conf <= b) if a > 0.5 else (conf >= a) & (conf <= b)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return e

sens, spec = tp / (tp + fn), tn / (tn + fp)
ece = ece_score(y, p)
summary = {
    "images": int(len(y)), "AUC": round(float(auc), 3), "AUC_95CI": f"{lo:.3f}-{hi:.3f}",
    "AUC_fold_mean_sd": f"{folds.test_auc.mean():.3f} ± {folds.test_auc.std():.3f}",
    "sensitivity": round(float(sens), 3), "specificity": round(float(spec), 3), "ECE": round(float(ece), 3),
    "TP": int(tp), "FN": int(fn), "TN": int(tn), "FP": int(fp)}
for k_, v in summary.items():
    print(f"{k_:>18}: {v}")

## 9. ROC curve and calibration plot

In [ ]:
import matplotlib.pyplot as plt
os.makedirs("/content/results", exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
fpr, tpr, _ = roc_curve(y, p)
ax[0].plot(fpr, tpr, lw=2, label=f"AUC {auc:.3f} (95% CI {lo:.3f}-{hi:.3f})")
ax[0].plot([0, 1], [0, 1], "--", color="grey")
ax[0].set_xlabel("False positive rate (1 - specificity)"); ax[0].set_ylabel("True positive rate (sensitivity)")
ax[0].set_title("ROC, 5-fold out-of-fold predictions"); ax[0].legend(loc="lower right")

bins = np.linspace(0, 1, 11); mids, fracs = [], []
for a, b in zip(bins[:-1], bins[1:]):
    m = (p >= a) & (p < b) if b < 1 else (p >= a) & (p <= b)
    if m.sum() >= 3:
        mids.append(p[m].mean()); fracs.append(y[m].mean())
ax[1].plot([0, 1], [0, 1], "--", color="grey", label="Perfect calibration")
ax[1].plot(mids, fracs, "o-", lw=2, label=f"Model (ECE {ece:.3f})")
ax[1].set_xlabel("Predicted probability of malignancy"); ax[1].set_ylabel("Observed fraction malignant")
ax[1].set_title("Calibration"); ax[1].legend(loc="upper left")
plt.tight_layout(); plt.savefig("/content/results/roc_calibration.png", dpi=150); plt.show()

## 10. Grad-CAM: where does the model look?
Check that the heat falls on the lesion, not on teeth, lips, retractors or the image border.
If it often lights up something other than the lesion, that is a shortcut, and it is worth reporting.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

model = models[-1]; model.eval()

te = df[df["fold"] == N_FOLDS - 1]
picks = pd.concat([te[te.label == 1].nlargest(3, "oof_prob"), te[te.label == 0].nsmallest(2, "oof_prob"),
                   te.assign(err=(te.oof_prob - te.label).abs()).nlargest(3, "err")]).drop_duplicates("path").head(8)
cam = GradCAM(model=model, target_layers=[model.blocks[-1]])
fig, axes = plt.subplots(2, len(picks), figsize=(2.6 * len(picks), 5.4))
for c, (_, r) in enumerate(picks.iterrows()):
    img = Image.open(r.path).convert("RGB").resize((IMG, IMG))
    rgb = np.asarray(img, dtype=np.float32) / 255
    g = cam(input_tensor=eval_tf(img).unsqueeze(0).to(DEVICE), targets=[ClassifierOutputTarget(0)])[0]
    axes[0, c].imshow(rgb); axes[1, c].imshow(show_cam_on_image(rgb, g, use_rgb=True))
    tag = "malignant" if r.label == 1 else "benign"
    axes[0, c].set_title(f"{tag}\np={r.oof_prob:.2f}", fontsize=9)
    for a in axes[:, c]: a.axis("off")
plt.tight_layout(); plt.savefig("/content/results/gradcam.png", dpi=150); plt.show()

## 11. Save results and print your resume and SOP lines

In [ ]:
import json
folds.to_csv("/content/results/folds.csv", index=False)
df[["path", "label", "group", "fold", "oof_prob", "fold_thr"]].to_csv("/content/results/oof_predictions.csv", index=False)
json.dump(summary, open("/content/results/summary.json", "w"), indent=2)
torch.save(models[-1].state_dict(), "/content/results/mobilenetv3_small_oral_fold5.pt")
shutil.make_archive("/content/oral_results", "zip", "/content/results")

print("RESUME LINE:")
print(f"• Trained MobileNetV3-Small on {summary['images']} oral lesion photos (Mendeley, CC BY 4.0) with 5-fold, "
      f"duplicate-grouped cross-validation: AUC {summary['AUC']} (95% CI {summary['AUC_95CI']}), "
      f"sensitivity {summary['sensitivity']*100:.1f}%, specificity {summary['specificity']*100:.1f}%, ECE {summary['ECE']}.")
print()
print("Send me the numbers in the box above and the zip below, and I will update the SOP and resume.")
try:
    from google.colab import files
    files.download("/content/oral_results.zip")
except Exception as e:
    print("Download oral_results.zip from the Files panel on the left.", e)

## Limits to state honestly
- 323 images from one region (Karnataka), with no patient IDs: grouping by image hash reduces, but cannot rule out, the same patient appearing in different folds.
- Labels are benign vs malignant at image level; the dataset does not say how each label was confirmed (biopsy or clinical judgement).
- This is single-visit classification. The app's real task, measuring change across visits, needs longitudinal photos, which no public dataset provides yet.